In [1]:
import os
import sys
import shutil
import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F


from tqdm import tqdm, trange

import warnings


from ml_collections import ConfigDict

In [2]:
sys.path.append('..')

In [3]:
from CRT_utils.CRT.core.model import Model
# from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.littlehelper import *

# from CRT_utils.CRT.test import test
from CRT_utils.CRT.test_zero_context_zero_target import test

# from CRT_utils.CRT.test_visual_search import test
# from utils.evaluate_uncertainty import evaluate_uncertainty
from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.core.dataset import COCODataset, COCODatasetWithID, COCODatasetMixOR, COCODatasetFullMix
from CRT_utils.CRT.core.model import Model
# from CRT_utils.CRT.core.metrics import AccuracyLogger 

In [4]:
config_dict = ConfigDict()


config_dict['config']                = None
config_dict['outdir']                = '../CRT_utils/CRT_weights_and_config/CRT_COCO_random'
config_dict['checkpoint']            = '../CRT_utils/CRT_weights_and_config/CRT_COCO_random/checkpoint_30.tar'

config_dict['annotations']           = '../datasets/COCO18_dset_for_CRT_training/train_metadata.json'
config_dict['imagedir']              = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt'

config_dict['test_annotations']      = '../datasets/COCO18_dset_for_CRT_training/coco18_test.json'
config_dict['test_imagedir']         = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test'
config_dict['test_frequency']        = 1

config_dict['epochs']                = 30
config_dict['save_frequency']        = 1
config_dict['print_batch_metrics']   = None

config_dict['batch_size']            = 32
config_dict['learning_rate']         = None
config_dict['imbalance_reweighting'] = None
config_dict['num_decoder_heads']     = None
config_dict['num_decoder_layers']    = 6
config_dict['uncertainty_gate_type'] = None
config_dict['uncertainty_threshold'] = 0
config_dict['weighted_prediction']   = None

In [5]:
cfg = create_config(config_dict)

In [6]:
dataset = COCODatasetFullMix(
    cfg.annotations, 
    cfg.imagedir, 
    image_size      = (224,224), 
    # _____ ORIGINAL CODE _____
    normalize_means = [0.485, 0.456, 0.406], 
    normalize_stds  = [0.229, 0.224, 0.225],
    # _____ ORIGINAL CODE _____
    
    # _____ MODIFIED VERSION _____
#     normalize_means = [0.5, 0.5, 0.5], 
#     normalize_stds  = [0.5, 5, 0.5]
    # _____ MODIFIED VERSION _____
    
    # _____ ADDED CODE _____
    category_dic_dir = '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl'
    
    # _____ ADDED CODE _____
    
)

dataloader = DataLoader(
    dataset, 
    batch_size  = cfg.batch_size, 
    num_workers = 1, 
    shuffle     = True, 
    pin_memory  = True, 
    drop_last   = True
)


-------------------------------
Annotation Counts
-------------------------------
potted plant               8652
tv                         5805
bottle                    24342
chair                     38491
car                       43867
stop sign                  1983
clock                      6334
cup                       20650
fork                       5479
knife                      7770
bowl                      14358
toilet                     4157
laptop                     4970
mouse                      2262
keyboard                   2855
microwave                  1673
oven                       3334
sink                       5610
Total                    202592
-------------------------------



In [7]:
NUM_CLASSES     = dataset.NUM_CLASSES
cfg.num_classes = NUM_CLASSES

os.makedirs(config_dict.outdir, exist_ok=True)

save_config(cfg, config_dict.outdir)

print(cfg)

annotations: ../datasets/COCO18_dset_for_CRT_training/train_metadata.json
batch_size: 32
checkpoint: ../CRT_utils/CRT_weights_and_config/CRT_COCO_random/checkpoint_30.tar
git: 61c90b997c92f042201e20c7b687b17e0ed05e7e
imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt
imbalance_reweighting: false
learning_rate: 1.0e-05
num_classes: 18
num_decoder_heads: 8
num_decoder_layers: 6
test_annotations: ../datasets/COCO18_dset_for_CRT_training/coco18_test.json
test_imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test
uncertainty_gate_type: learned
uncertainty_threshold: 0
weighted_prediction: false



In [8]:
model = Model.from_config(cfg)

In [10]:
test(
    model, 
    cfg.test_annotations, 
    cfg.test_imagedir, 
#             '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl',
    outdir = '../CRT_utils/CRT_weights_and_config/debug',
    epoch  = 1,
)

-------------------------------
Annotation Counts
-------------------------------
chair                        50
fork                         46
sink                         55
tv                           56
bowl                         28
car                          20
clock                        23
cup                          55
keyboard                     36
knife                        28
laptop                       24
mouse                        21
oven                         20
potted plant                 30
toilet                       31
bottle                       33
stop sign                    25
microwave                    31
Total                       612
-------------------------------



Test Batches:   0%|                                                                                                                      | 0/612 [00:01<?, ?it/s]


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/Users/nguyentuan/Mirror/STUDY_DOCUMENTS/FYP/TCT code/TCT-visual-search/TCT notebooks/../CRT_utils/CRT/core/dataset.py", line 515, in __getitem__
    image, _, bbox_relative, labe

In [11]:
import json

In [ ]:
with open('')